## Phase 1 — Environment Bootstrap

In [ ]:
# Cell 0 – GPU Check
# Run this first. If CUDA is False, go to: Runtime > Change runtime type > T4 GPU
import torch

print("CUDA available :", torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("GPU            :", torch.cuda.get_device_name(0))
    print("VRAM           :", round(props.total_memory / 1024**3, 1), "GB")
    print("PyTorch        :", torch.__version__)

else:
    print()
    print("⚠️  No GPU detected.")
    print("   Go to: Runtime > Change runtime type > Hardware accelerator > T4 GPU")
    print("   Save, wait for reconnect, then re-run this cell.")

CUDA available : True
GPU            : Tesla T4
VRAM           : 14.6 GB
PyTorch        : 2.10.0+cu128


In [ ]:
# Cell 1 – Mount Google Drive
# All checkpoints, data, and outputs are stored on Drive so they persist
# across Colab session resets.
# from google.colab import drive
# import os

# drive.mount('/content/drive')

# DRIVE_ROOT  = "/content/drive/MyDrive/BUyen_Qnhu++/src"
# DRIVE_DATA  = os.path.join(DRIVE_ROOT, "data")
# DRIVE_CKPTS = os.path.join(DRIVE_ROOT, "ckpts")
# DRIVE_OUT   = os.path.join(DRIVE_ROOT, "out")

# for d in [DRIVE_DATA, DRIVE_CKPTS, DRIVE_OUT]:
#     os.makedirs(d, exist_ok=True)

# print("Drive mounted. Persistent folders created:")
# print(f"  Data        → {DRIVE_DATA}")
# print(f"  Checkpoints → {DRIVE_CKPTS}")
# print(f"  Output      → {DRIVE_OUT}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
!sudo apt update


Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [89.0 kB]
Get:5 https://cli.github.com/packages stable/main amd64 Packages [354 B]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,533 kB]
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,970 kB]
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.0 MB]
Hit:11 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:12 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:13 https:/

In [4]:
!sudo apt install python3.10 python3.10-venv python3.10-dev

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
python3.10 is already the newest version (3.10.12-1~22.04.15).
python3.10 set to manually installed.
The following additional packages will be installed:
  python3-pip-whl python3-setuptools-whl
The following NEW packages will be installed:
  python3-pip-whl python3-setuptools-whl python3.10-dev python3.10-venv
0 upgraded, 4 newly installed, 0 to remove and 86 not upgraded.
Need to get 2,989 kB of archives.
After this operation, 3,277 kB of additional disk space will be used.
Get:1 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy/main amd64 python3-setuptools-whl all 68.1.2-2~jammy3 [792 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 python3-pip-whl all 22.0.2+dfsg-1ubuntu0.7 [1,683 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 python3.10-dev amd64 3.10.12-1~22.04.15 [508 kB]
Get:4 http://archive.ubuntu.com/ubuntu jammy-updates/unive

In [ ]:
# No venv needed on vast.ai — system Python already has PyTorch 2.1
print("Skipping venv creation — using system Python")

In [ ]:
# No venv pip upgrade needed
print("Skipping venv pip upgrade")

In [7]:
# Packages are installed automatically by --onstart-cmd when the instance boots.
# This cell verifies they are ready before continuing.
import subprocess, sys

result = subprocess.run(
    [sys.executable, '-c',
     'import transformers, sacrebleu, tokenizers, jupyterlab; print("All packages ready.")'],
    capture_output=True, text=True
)
if result.returncode == 0:
    print(result.stdout.strip())
else:
    print("Packages not ready yet — installing now (this takes ~2 min)...")
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q',
        'bert-score', 'blobfile', 'datasets', 'huggingface-hub==0.4.0',
        'mpi4py', 'nltk', 'pandas', 'protobuf', 'rouge-score', 'sacrebleu',
        'sacremoses', 'scikit-learn', 'scipy', 'spacy', 'tokenizers',
        'torchmetrics', 'tqdm', 'transformers==4.18.0', 'jupyterlab', 'ipykernel'
    ], check=True)
    print("Done.")


Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu113
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 GB 598.7 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.3/22.3 MB 62.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 109.5 MB/s eta 0:00:00
  Obtaining dependency information for typing-extensions from https://download.pytorch.org/whl/typing_extensions-4.15.0-py3-none-any.whl.metadata
Discarding https://download.pytorch.org/whl/typing_extensions-4.15.0-py3-none-any.whl (from https://download.pytorch.org/whl/cu113/typing-extensions/): Requested typing-extensions from https://download.pytorch.org/whl/typing_extensions-4.15.0-py3-none-any.whl (from torch==1.11.0+cu113) has inconsistent Name: expected 'typing-extensions', but metadata has 'typing_extensions'
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 kB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 5.7 MB/s eta 

## Phase 2 — Data Setup

**Before running Cell 4**, upload your 6 parallel data files to Google Drive at:

```
MyDrive/SeqDiffuSeq_ZH_VI/data/
```

| File | Description |
|------|-------------|
| `train_augmented.zh` | Training source (Chinese) |
| `train_augmented.vi` | Training target (Vietnamese) |
| `valid.zh` | Validation source |
| `valid.vi` | Validation target |
| `test.zh` | Test source |
| `test.vi` | Test target |

You can upload via:
- **drive.google.com** → navigate to `SeqDiffuSeq_ZH_VI/data/` → drag-and-drop files
- **Colab file browser** → left sidebar → Files icon → navigate to the Drive path → right-click → Upload

In [ ]:
import os, subprocess, glob, shutil

REPO_DIR     = "/root/SeqDiffuSeq"
DRIVE_DATA   = "/root/train_dataset"
DST_DATA_DIR = os.path.join(REPO_DIR, "data", "en_zh")
CKPT_DIR     = os.path.join(REPO_DIR, "ckpts", "en_zh")
OUT_DIR      = os.path.join(CKPT_DIR, "inference_out")
VENV_PYTHON  = "python3"   # vast.ai system Python (PyTorch 2.1 already installed)

In [35]:

required_files = [
    "train_clean.en", "train_clean.zh",
    "valid_clean.en", "valid_clean.zh",
    "test_clean.en",  "test_clean.zh",
]

all_ok = True
for fname in required_files:
    path = os.path.join(DRIVE_DATA, fname)
    if os.path.exists(path):
        n = sum(1 for _ in open(path, encoding='utf-8'))
        print(f"  \u2713  {fname:30s}  {n:>8,} lines")
    else:
        print(f"  \u2717  {fname:30s}  NOT FOUND")
        all_ok = False


  ✓  train.en                         240,192 lines
  ✓  train.zh                         240,213 lines
  ✓  valid.en                          30,025 lines
  ✓  valid.zh                          30,041 lines
  ✓  test.en                           30,018 lines
  ✓  test.zh                           30,016 lines


In [36]:
os.makedirs(DST_DATA_DIR, exist_ok=True)

# Copy cleaned files; destination keeps original names so training scripts need no changes.
file_mapping = {
    "train_clean.en": "train.en",
    "train_clean.zh": "train.zh",
    "valid_clean.en": "valid.en",
    "valid_clean.zh": "valid.zh",
    "test_clean.en":  "test.en",
    "test_clean.zh":  "test.zh",
}

print(f"Copying to: {DST_DATA_DIR}\n")
for src_name, dst_name in file_mapping.items():
    src = os.path.join(DRIVE_DATA, src_name)
    dst = os.path.join(DST_DATA_DIR, dst_name)
    if not os.path.exists(src):
        raise FileNotFoundError(f"Missing: {src} — run clean_dataset.py first.")
    shutil.copy2(src, dst)
    n = sum(1 for _ in open(dst, encoding='utf-8'))
    print(f"  Copied  {src_name} -> {dst_name:20s}  ({n:,} lines)")

print(f"\nData ready. Repo data dir:")
print(f"   {sorted(os.listdir(DST_DATA_DIR))}")


Copying to: /content/drive/MyDrive/BUyen_Qnhu++/src/SeqDiffuSeq/data/en_zh

  Copied  train.en                        (240,192 lines)
  Copied  train.zh                        (240,213 lines)
  Copied  valid.en                        (30,025 lines)
  Copied  valid.zh                        (30,041 lines)
  Copied  test.en                         (30,018 lines)
  Copied  test.zh                         (30,016 lines)
Data ready. Repo data dir contents:
   ['test.en', 'test.zh', 'train.en', 'train.zh', 'valid.en', 'valid.zh']


## Phase 3 — Tokenizer Training

In [ ]:
# Train byte-level BPE tokenizer on the en+zh corpus.
# Saves vocab.json and merges.txt to data/en_zh/.
# Always retrains to pick up any vocab_size change.
import shutil

for fname in ["vocab.json", "merges.txt"]:
    fpath = os.path.join(DST_DATA_DIR, fname)
    if os.path.exists(fpath):
        os.remove(fpath)
        print(f"Removed old tokenizer file: {fname}")

result = subprocess.run(
    [VENV_PYTHON, "tokenizer_utils.py", "train-byte-level", "en_zh", "32000"],
    cwd=REPO_DIR, capture_output=True, text=True
)
print(result.stdout[-2000:])
if result.returncode != 0:
    print("STDERR:", result.stderr[-1000:])
else:
    print("Tokenizer trained (vocab=32000) and saved to", DST_DATA_DIR)

## Phase 4 — Training

In [39]:
# Download facebook/bart-base using the notebook's own Python (reliable internet in Colab).
# Saves config + tokenizer to Drive so subprocesses load them offline.
PRETRAINED_DIR = os.path.join(REPO_DIR, "pretrained", "bart-base")
CONFIG_NAME    = PRETRAINED_DIR

if os.path.exists(os.path.join(PRETRAINED_DIR, "config.json")):
    print("bart-base already cached →", PRETRAINED_DIR)
else:
    os.makedirs(PRETRAINED_DIR, exist_ok=True)
    from transformers import BartConfig, BartTokenizerFast
    BartConfig.from_pretrained("facebook/bart-base").save_pretrained(PRETRAINED_DIR)
    BartTokenizerFast.from_pretrained("facebook/bart-base").save_pretrained(PRETRAINED_DIR)
    print("bart-base saved to", PRETRAINED_DIR)


bart-base already cached → /content/drive/MyDrive/BUyen_Qnhu++/src/SeqDiffuSeq/pretrained/bart-base


In [ ]:
os.makedirs(os.path.join(CKPT_DIR, "log"), exist_ok=True)

args = [
VENV_PYTHON, "-u", "main.py",
"--checkpoint_path", CKPT_DIR,
"--src", "en",
"--tgt", "zh",
"--train_txt_path", "./data/en-zh/train",
"--val_txt_path", "./data/en-zh/valid",
"--dataset", "en-zh",
"--config_name", CONFIG_NAME,
"--diffusion_steps", "1000",
"--noise_schedule", "sqrt",
"--sequence_len", "64",
"--sequence_len_src", "128",
"--batch_size", "16",
"--lr", "1e-4",
"--lr_anneal_steps", "100000",
"--resume_checkpoint", "",
"--warmup", "500",
"--save_interval", "5000",
"--eval_interval", "2500",
"--log_interval", "100",
"--schedule_update_stride", "200",
"--loss_update_granu", "20",
"--encoder_layers", "6",
"--decoder_layers", "6",
"--num_heads", "8",
"--in_channel", "512",
"--out_channel", "512",
"--num_channels", "2048",
"--vocab_size", "32005",
"--dropout", "0.3",
"--predict_xstart", "True",
"--seed", "42",
"--schedule_sampler", "uniform",
"--init_pretrained", "True",
"--freeze_embeddings", "False",
"--use_pretrained_embeddings", "False",
]

env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"]    = "0"
env["DIFFUSION_BLOB_LOGDIR"]   = os.path.join(CKPT_DIR, "log")
env["TRANSFORMERS_OFFLINE"]    = "1"

print("Starting training… logs → ", os.path.join(CKPT_DIR, "log"))
result = subprocess.run(args, cwd=REPO_DIR, env=env, capture_output=True, text=True)
print(result.stdout[-3000:])
if result.returncode != 0:
    print("STDERR:", result.stderr[-2000:])
    print("Exit code:", result.returncode)

## Phase 5 — Inference

In [ ]:
import numpy as np

os.makedirs(OUT_DIR, exist_ok=True)

# Pick the latest EMA checkpoint
ema_ckpts = sorted(glob.glob(os.path.join(CKPT_DIR, "ema_0.9999_*.pt")))
if not ema_ckpts:
    raise FileNotFoundError("No EMA checkpoint found — train first.")
MODEL_PATH = ema_ckpts[-1]
print(f"Model  : {MODEL_PATH}")

# Pick the latest noise-schedule .npy; generate the initial one as fallback if missing
npy_files = sorted(glob.glob(os.path.join(CKPT_DIR, "alpha_cumprod_step_*.npy")))
if npy_files:
    NPY_PATH = npy_files[-1]
    print(f"Sched  : {NPY_PATH}")
else:
    print("No updated schedule found — generating initial sqrt alpha_cumprod as fallback.")
    T, S = 1000, 64
    alpha_bar = lambda t: 1 - np.sqrt(t + 0.0001)
    betas = np.array([min(1 - alpha_bar((i+1)/T) / alpha_bar(i/T), 0.999) for i in range(T)])
    alphas = np.tile((1.0 - betas)[:, None], (1, S))
    alphas_cumprod = np.cumprod(alphas, axis=0)
    NPY_PATH = os.path.join(CKPT_DIR, "alpha_cumprod_step_0_initial.npy")
    np.save(NPY_PATH, alphas_cumprod)
    print(f"Saved fallback schedule → {NPY_PATH}")

# Count full test set so inference runs on every sentence
_test_src = os.path.join(DST_DATA_DIR, "test.en")
with open(_test_src, encoding="utf-8") as _f:
    num_test_sentences = sum(1 for _ in _f)
print(f"Running inference on full test set: {num_test_sentences:,} sentences")

args = [
    VENV_PYTHON, "-u", "inference_main.py",
    "--model_name_or_path", MODEL_PATH,
    "--batch_size",          "16",
    "--num_samples",         str(num_test_sentences),
    "--mbr_sample",          "1",
    "--out_dir",             OUT_DIR,
    "--time_schedule_path",  NPY_PATH,
    "--src",                 "en",
    "--tgt",                 "zh",
    "--sequence_len",        "64",
    "--sequence_len_src",    "128",
    "--diffusion_steps",     "200",
    "--noise_schedule",      "sqrt",
    "--use_ddim",            "True",
    "--dataset",             "en-zh",
    "--val_txt_path",        "./data/en-zh/test",
    "--verbose",             "yes",
    "--clamp",               "clamp",
    "--vocab_size",          "32005",
]

env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"]   = "0"
env["TRANSFORMERS_OFFLINE"]   = "1"
env["TOKENIZERS_PARALLELISM"] = "false"

result = subprocess.run(args, cwd=REPO_DIR, env=env, capture_output=True, text=True)
print(result.stdout[-3000:])
if result.returncode != 0:
    print("STDERR:", result.stderr[-3000:])
print("Exit code:", result.returncode)
print("Results saved to:", OUT_DIR)

## Phase 6 — BLEU Evaluation

In [49]:
# Find the decoded output file (written to CKPT_DIR alongside the checkpoint)
decoded_files = sorted([
    f for f in glob.glob(os.path.join(CKPT_DIR, "ema_*.pt.samples_*.txt"))
    if "raw-output-ids" not in f
])

if not decoded_files:
    print("No decoded output file found in:", CKPT_DIR)
    print("Files present:")
    for f in glob.glob(os.path.join(CKPT_DIR, "*.txt")):
        print(" ", f)
else:
    output_file  = decoded_files[-1]
    # Short name: extract step number, e.g. "001000" -> "eval_step001000"
    import re
    m = re.search(r'ema_[\d.]+_(\d+)', os.path.basename(output_file))
    short = f"eval_step{m.group(1)}" if m else "eval"
    os.makedirs(OUT_DIR, exist_ok=True)
    eval_csv     = os.path.join(OUT_DIR, short + ".csv")
    eval_summary = os.path.join(OUT_DIR, short + "_summary.txt")
    print(f"Evaluating: {output_file}\n")

    eval_script = f"""
import json, csv, sacrebleu

with open({repr(output_file)}, "r", encoding="utf-8") as f:
    pairs = [json.loads(l.strip()) for l in f if l.strip()]

hypotheses = [p[0] for p in pairs]
references  = [p[1] for p in pairs]

src_path = {repr(os.path.join(REPO_DIR, "data", "en-zh", "test.en"))}
with open(src_path, "r", encoding="utf-8") as f:
    sources = [l.strip() for l in f if l.strip()]

n = min(len(sources), len(hypotheses))
sources, hypotheses, references = sources[:n], hypotheses[:n], references[:n]

bleu_char = sacrebleu.corpus_bleu(hypotheses, [references], tokenize='char')
bleu_13a  = sacrebleu.corpus_bleu(hypotheses, [references], tokenize='13a')

with open({repr(eval_csv)}, "w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["source_zh", "hypothesis_vi", "reference_vi"])
    for src, hyp, ref in zip(sources, hypotheses, references):
        writer.writerow([src, hyp, ref])

summary_text = "\\n".join([
    f"Output file     : {repr(output_file)}",
    f"CSV             : {repr(eval_csv)}",
    f"Num samples     : {{n}}",
    "",
    f"SacreBLEU (char): {{bleu_char.score:.2f}}",
    f"SacreBLEU (13a) : {{bleu_13a.score:.2f}}",
])
print(summary_text)
with open({repr(eval_summary)}, "w", encoding="utf-8") as f:
    f.write(summary_text)
print(f"\\nCSV     saved to: {repr(eval_csv)}")
print(f"Summary saved to: {repr(eval_summary)}")
"""

    result = subprocess.run(
        [VENV_PYTHON, "-c", eval_script],
        capture_output=True, text=True
    )
    print(result.stdout)
    if result.returncode != 0:
        print("STDERR:", result.stderr[-2000:])


Evaluating: /content/drive/MyDrive/BUyen_Qnhu++/src/SeqDiffuSeq/ckpts/en_zh/ema_0.9999_030000.pt.samples_50.steps-1000.clamp-clamp-normal_101.txt

Output file     : '/content/drive/MyDrive/BUyen_Qnhu++/src/SeqDiffuSeq/ckpts/en_zh/ema_0.9999_030000.pt.samples_50.steps-1000.clamp-clamp-normal_101.txt'
CSV             : '/content/drive/MyDrive/BUyen_Qnhu++/src/SeqDiffuSeq/ckpts/en_zh/inference_out/eval_step030000.csv'
Num samples     : 50

SacreBLEU (char): 0.23
SacreBLEU (13a) : 0.04

CSV     saved to: '/content/drive/MyDrive/BUyen_Qnhu++/src/SeqDiffuSeq/ckpts/en_zh/inference_out/eval_step030000.csv'
Summary saved to: '/content/drive/MyDrive/BUyen_Qnhu++/src/SeqDiffuSeq/ckpts/en_zh/inference_out/eval_step030000_summary.txt'

